# COMET Evaluation for Final Analysis (Idioms + WMT)

This notebook extends the earlier COMET workflow so that it scores and summarizes:

- the **canonical 25-example `idioms_test` subset**
- the **canonical 25-example `wmt_test` subset**

It uses the wide comparison CSV from notebook 20 as the source of truth for the
canonical comparison sets, then filters FLAN prediction files down to those exact
`(src, ref)` pairs before scoring with COMET.

## 0. Install note

Run installs in a fresh Colab runtime if needed, then restart the runtime manually.

In [14]:
# Optional install cell for a fresh Colab runtime.
# Uncomment and run only if COMET is not installed, then restart runtime manually.
#!pip -q install -U unbabel-comet
# print("Restart the runtime now, then continue.")

In [15]:
from comet import download_model, load_from_checkpoint
import torch
import os
from pathlib import Path
import pandas as pd
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

device: cpu


## 1. Paths and configuration

In [16]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# Drive / project paths
PROJECT_ROOT = "/content/drive/MyDrive/ds266_idiom_mt"
QUAL_PREDS_DIR = os.path.join(PROJECT_ROOT, "qual_preds")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")

OUT_DIR = os.path.join(QUAL_PREDS_DIR, "comet_eval")
os.makedirs(OUT_DIR, exist_ok=True)

# Wide CSV input (from 20_preds notebook)
WIDE_CSV_PATH = os.path.join(QUAL_PREDS_DIR, "preds_5models.csv")

# Per-model inputs from FLAN-T5 prompting notebook.
# Idioms: keep the prompting experiment outputs.
# WMT: use the new 0-shot WMT25 file.
MODEL_FILES = {
    "flan_t5_small_prompt_0shot": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_0shot_preds.csv"),
    "flan_t5_small_prompt_3shot": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_3shot_preds.csv"),
    "flan_t5_small_prompt_5shot": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_5shot_preds.csv"),
        "flan_t5_small_prompt_0shot_wmt25": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_0shot_wmt25_preds.csv"),
    "flan_t5_small_prompt_3shot_wmt25": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_3shot_wmt25_preds.csv"),
    "flan_t5_small_prompt_5shot_wmt25": os.path.join(RESULTS_DIR, "flan_t5_small_prompt_5shot_wmt25_preds.csv"),
}

# Optional metrics merge
METRICS_FILES = [
    os.path.join(RESULTS_DIR, "metrics.csv"),
    os.path.join(RESULTS_DIR, "prompting_metrics.csv"),
    os.path.join(RESULTS_DIR, "prompting", "prompting_metrics.csv"),
]

# COMET model
COMET_MODEL_NAME = "Unbabel/wmt22-comet-da"
BATCH_SIZE = 8

# Canonical evaluation set definitions from the wide CSV
TARGET_COUNTS = {
    "idioms_test": 25,
    "wmt_test": 25,
}
STRICT_CANONICAL_SIZE = False  # set True if you want a hard failure when counts do not match

## 2. Helpers

In [18]:
def existing_path(p):
    return p is not None and os.path.exists(p)

def normalize_text_value(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).strip().split())

def infer_group_from_row(row):
    for key in ("group", "split", "dataset", "eval_group"):
        if key in row and pd.notna(row[key]):
            return str(row[key])
    return "unknown"

def normalize_wide_df(df):
    src_col = None
    ref_col = None
    for c in ["src", "source", "english", "input"]:
        if c in df.columns:
            src_col = c
            break
    for c in ["tgt", "reference", "ref", "german", "target"]:
        if c in df.columns:
            ref_col = c
            break
    if src_col is None or ref_col is None:
        raise ValueError(f"Could not infer src/ref columns from wide CSV columns: {list(df.columns)}")

    pred_cols = [c for c in df.columns if c.startswith("pred_")]
    if not pred_cols:
        raise ValueError("No pred_* columns found in wide CSV.")

    rows = []
    for row_idx, r in df.iterrows():
        src = r[src_col]
        ref = r[ref_col]
        group = infer_group_from_row(r)
        example_id = r["id"] if "id" in df.columns else row_idx
        for pc in pred_cols:
            pred = r[pc]
            model = pc.replace("pred_", "")
            rows.append({
                "example_id": example_id,
                "src": src,
                "ref": ref,
                "pred": pred,
                "model": model,
                "group": group,
                "input_source": "wide_csv",
            })
    return pd.DataFrame(rows)

def normalize_per_model_df(df, model_name):
    src_col = None
    ref_col = None
    pred_col = None

    for c in ["src", "source", "english", "input"]:
        if c in df.columns:
            src_col = c
            break
    for c in ["tgt", "reference", "ref", "german", "target"]:
        if c in df.columns:
            ref_col = c
            break
    for c in ["pred", "prediction", "mt"]:
        if c in df.columns:
            pred_col = c
            break

    if src_col is None or ref_col is None or pred_col is None:
        raise ValueError(f"Could not infer src/ref/pred columns for {model_name} from columns: {list(df.columns)}")

    # Determine group
    if any(k in df.columns for k in ["group", "split", "dataset", "eval_group"]):
        for c in ["group", "split", "dataset", "eval_group"]:
            if c in df.columns:
                group_series = df[c].fillna("unknown").astype(str)
                break
    else:
        # Infer from model file name / expected usage
        if "wmt25" in model_name.lower():
            group_series = pd.Series(["wmt_test"] * len(df), index=df.index)
        else:
            group_series = pd.Series(["idioms_test"] * len(df), index=df.index)

    example_ids = df["id"] if "id" in df.columns else df.index

    return pd.DataFrame({
        "example_id": example_ids,
        "src": df[src_col],
        "ref": df[ref_col],
        "pred": df[pred_col],
        "model": model_name,
        "group": group_series,
        "input_source": "per_model_csvs",
    })

def build_canonical_pairs_by_group(wide_long_df, target_counts):
    canonical_by_group = {}
    canonical_parts = []

    for group_name, expected_n in target_counts.items():
        subset = (
            wide_long_df.loc[wide_long_df["group"] == group_name, ["example_id", "src", "ref", "group"]]
            .drop_duplicates()
            .copy()
        )
        subset["src_norm"] = subset["src"].map(normalize_text_value)
        subset["ref_norm"] = subset["ref"].map(normalize_text_value)
        subset["pair_key"] = list(zip(subset["src_norm"], subset["ref_norm"]))

        print(f"Canonical group '{group_name}' unique examples:", len(subset))
        if STRICT_CANONICAL_SIZE and len(subset) != expected_n:
            raise ValueError(
                f"Canonical set size for {group_name} was {len(subset)} but expected {expected_n}."
            )

        canonical_by_group[group_name] = set(subset["pair_key"].tolist())
        canonical_parts.append(subset)

    canonical_df = pd.concat(canonical_parts, ignore_index=True)
    return canonical_df, canonical_by_group

def filter_to_canonical_by_group(df, canonical_by_group):
    x = df.copy()
    x["src_norm"] = x["src"].map(normalize_text_value)
    x["ref_norm"] = x["ref"].map(normalize_text_value)
    x["pair_key"] = list(zip(x["src_norm"], x["ref_norm"]))

    keep_mask = []
    for _, r in x.iterrows():
        group_name = r["group"]
        keep_mask.append(group_name in canonical_by_group and r["pair_key"] in canonical_by_group[group_name])

    x = x.loc[keep_mask].copy()
    return x

def validate_model_coverage(filtered_df):
    coverage = (
        filtered_df.groupby(["group", "model"]).size().reset_index(name="n")
        .sort_values(["group", "model"])
        .reset_index(drop=True)
    )
    print("Rows per model after apples-to-apples filtering:")
    display(coverage)
    return coverage

def load_inputs():
    parts = []

    if not existing_path(WIDE_CSV_PATH):
        raise FileNotFoundError(f"Wide CSV not found: {WIDE_CSV_PATH}")

    wide_df = pd.read_csv(WIDE_CSV_PATH)
    wide_long = normalize_wide_df(wide_df)
    print("Loaded wide CSV:", WIDE_CSV_PATH)

    canonical_df, canonical_by_group = build_canonical_pairs_by_group(wide_long, TARGET_COUNTS)
    wide_long = filter_to_canonical_by_group(wide_long, canonical_by_group)
    print("Wide CSV rows after canonical filtering:", len(wide_long))
    parts.append(wide_long)

    model_parts = []
    for model_name, path in MODEL_FILES.items():
        if not existing_path(path):
            print(f"Skipping missing per-model file for {model_name}: {path}")
            continue
        df = pd.read_csv(path)
        model_df = normalize_per_model_df(df, model_name)
        before_n = len(model_df)
        model_df = filter_to_canonical_by_group(model_df, canonical_by_group)
        print(f"Loaded per-model CSV: {path} | kept {len(model_df)} of {before_n} rows after canonical filtering")
        model_parts.append(model_df)

    if model_parts:
        parts.append(pd.concat(model_parts, ignore_index=True))

    if not parts:
        raise ValueError("No input data loaded. Check file paths.")

    combined = pd.concat(parts, ignore_index=True)
    combined["src"] = combined["src"].astype(str)
    combined["ref"] = combined["ref"].astype(str)
    combined["pred"] = combined["pred"].fillna("").astype(str)
    combined["group"] = combined["group"].fillna("unknown").astype(str)

    # Normalize FLAN 0-shot naming across idioms and WMT before any summaries/pivots.
    combined["model"] = combined["model"].replace({
        "flan_t5_small_prompt_0shot_wmt25": "flan_t5_small_prompt_0shot",
        "flan_t5_small_prompt_3shot_wmt25": "flan_t5_small_prompt_3shot",
        "flan_t5_small_prompt_5shot_wmt25": "flan_t5_small_prompt_5shot",
    })

    validate_model_coverage(combined)
    return combined, canonical_df

def score_with_comet(df, model, batch_size=8):
    data = [{"src": s, "mt": p, "ref": r} for s, p, r in zip(df["src"], df["pred"], df["ref"])]
    outputs = model.predict(data, batch_size=batch_size, gpus=1 if device=="cuda" else 0)
    scores = outputs.scores if hasattr(outputs, "scores") else outputs["scores"]
    out = df.copy()
    out["comet"] = scores
    return out

def summarize_scores(scored_df):
    overall = (
        scored_df.groupby("model", as_index=False)
        .agg(
            n=("comet", "size"),
            comet_mean=("comet", "mean"),
            comet_std=("comet", "std"),
            input_sources=("input_source", lambda s: ",".join(sorted(set(map(str, s)))))
        )
        .sort_values("comet_mean", ascending=False)
        .reset_index(drop=True)
    )

    by_group = (
        scored_df.groupby(["model", "group"], as_index=False)
        .agg(
            n=("comet", "size"),
            comet_mean=("comet", "mean"),
            comet_std=("comet", "std"),
            input_sources=("input_source", lambda s: ",".join(sorted(set(map(str, s)))))
        )
        .sort_values(["group", "comet_mean"], ascending=[True, False])
        .reset_index(drop=True)
    )

    by_input_source = (
        scored_df.groupby(["input_source", "model", "group"], as_index=False)
        .agg(
            n=("comet", "size"),
            comet_mean=("comet", "mean"),
            comet_std=("comet", "std"),
        )
        .sort_values(["group", "input_source", "comet_mean"], ascending=[True, True, False])
        .reset_index(drop=True)
    )

    pivot_by_group = by_group.pivot(index="model", columns="group", values="comet_mean").reset_index()
    pivot_by_input_source = (
        by_input_source.assign(input_group=by_input_source["input_source"] + "__" + by_input_source["group"])
        .pivot(index="model", columns="input_group", values="comet_mean")
        .reset_index()
    )
    return overall, by_group, by_input_source, pivot_by_group, pivot_by_input_source

def try_merge_metrics(overall_df):
    metric_frames = []
    for p in METRICS_FILES:
        if existing_path(p):
            try:
                x = pd.read_csv(p)
                x["metrics_source_file"] = os.path.basename(p)
                metric_frames.append(x)
            except Exception as e:
                print(f"Could not read metrics file {p}: {e}")

    if not metric_frames:
        print("No metrics files found for optional merge.")
        return None

    metrics = pd.concat(metric_frames, ignore_index=True, sort=False)
    if "run_name" in metrics.columns:
        right_key = "run_name"
    elif "model_name" in metrics.columns:
        right_key = "model_name"
    else:
        print("Metrics files do not contain run_name/model_name. Skipping merge.")
        return None

    merged = overall_df.merge(metrics, left_on="model", right_on=right_key, how="left")
    return merged

## 3. Load COMET model

In [19]:
model_path = download_model(COMET_MODEL_NAME)
comet_model = load_from_checkpoint(model_path)
print("Loaded COMET model:", COMET_MODEL_NAME)

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

INFO:pytorch_lightning.utilities.migration.utils:Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../root/.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Loaded COMET model: Unbabel/wmt22-comet-da


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py:197: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


## 4. Load and combine inputs (apples-to-apples idioms + WMT subsets)

In [20]:
combined_df, canonical_df = load_inputs()
print("Rows loaded:", len(combined_df))
print("Models:", sorted(combined_df["model"].unique().tolist()))
print("Groups:", sorted(combined_df["group"].dropna().unique().tolist()))
print("Input sources:", combined_df["input_source"].value_counts().to_dict())
canonical_df.head()

Loaded wide CSV: /content/drive/MyDrive/ds266_idiom_mt/qual_preds/preds_5models.csv
Canonical group 'idioms_test' unique examples: 25
Canonical group 'wmt_test' unique examples: 25
Wide CSV rows after canonical filtering: 250
Loaded per-model CSV: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_0shot_preds.csv | kept 25 of 100 rows after canonical filtering
Loaded per-model CSV: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_3shot_preds.csv | kept 25 of 100 rows after canonical filtering
Loaded per-model CSV: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_5shot_preds.csv | kept 25 of 100 rows after canonical filtering
Loaded per-model CSV: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_0shot_wmt25_preds.csv | kept 25 of 25 rows after canonical filtering
Loaded per-model CSV: /content/drive/MyDrive/ds266_idiom_mt/results/flan_t5_small_prompt_3shot_wmt25_preds.csv | kept 25 of 25 rows after canonical filtering

,group,model,n
0,idioms_test,baseline,25
1,idioms_test,flan_t5_small_prompt_0shot,25
2,idioms_test,flan_t5_small_prompt_3shot,25
3,idioms_test,flan_t5_small_prompt_5shot,25
4,idioms_test,idiom_only_v1,25
5,idioms_test,lora_r16_stage2,25
6,idioms_test,lora_r4_stage2,25
7,idioms_test,lora_r8_stage2,25
8,wmt_test,baseline,25
9,wmt_test,flan_t5_small_prompt_0shot,25


Rows loaded: 400
Models: ['baseline', 'flan_t5_small_prompt_0shot', 'flan_t5_small_prompt_3shot', 'flan_t5_small_prompt_5shot', 'idiom_only_v1', 'lora_r16_stage2', 'lora_r4_stage2', 'lora_r8_stage2']
Groups: ['idioms_test', 'wmt_test']
Input sources: {'wide_csv': 250, 'per_model_csvs': 150}


,example_id,src,ref,group,src_norm,ref_norm,pair_key
0,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...",idioms_test,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu..."
1,1,She's always trying to pass the buck and I'm s...,"Sie versucht immer, den schwarzen Peter weiter...",idioms_test,She's always trying to pass the buck and I'm s...,"Sie versucht immer, den schwarzen Peter weiter...",(She's always trying to pass the buck and I'm ...
2,2,He has a reputation as a loose cannon whose co...,"Er gilt als tickende Zeitbombe, dessen Äußerun...",idioms_test,He has a reputation as a loose cannon whose co...,"Er gilt als tickende Zeitbombe, dessen Äußerun...",(He has a reputation as a loose cannon whose c...
3,3,He's been working late with her every night th...,Er hat diese Woche jeden Abend mit ihr lange g...,idioms_test,He's been working late with her every night th...,Er hat diese Woche jeden Abend mit ihr lange g...,(He's been working late with her every night t...
4,4,Burning the candle at both ends is a recipe fo...,"Raubbau mit der Gesundheit zu treiben, ist ein...",idioms_test,Burning the candle at both ends is a recipe fo...,"Raubbau mit der Gesundheit zu treiben, ist ein...",(Burning the candle at both ends is a recipe f...


## 5. Score with COMET

In [21]:
scored_df = score_with_comet(combined_df, comet_model, batch_size=BATCH_SIZE)
scored_df.head()

INFO:pytorch_lightning.utilities.rank_zero:GPU available: False, used: False
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 50/50 [08:08<00:00,  9.77s/it]


,example_id,src,ref,pred,model,group,input_source,src_norm,ref_norm,pair_key,comet
0,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",baseline,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299
1,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Papa sagt, ich bin nicht für das Militär gesch...",idiom_only_v1,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.928583
2,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r4_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299
3,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r8_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299
4,0,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","Dad sagt, ich bin nicht für das Militär geeign...",lora_r16_stage2,idioms_test,wide_csv,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...","(Dad says I'm not cut out for the military, bu...",0.924299


## 6. Summaries

In [22]:
overall_df, by_group_df, by_input_source_df, pivot_by_group_df, pivot_by_input_source_df = summarize_scores(scored_df)

display(overall_df)
display(by_group_df)
display(pivot_by_group_df)
display(pivot_by_input_source_df)

,model,n,comet_mean,comet_std,input_sources
0,idiom_only_v1,50,0.807305,0.121235,wide_csv
1,lora_r16_stage2,50,0.789763,0.133208,wide_csv
2,lora_r8_stage2,50,0.787165,0.137748,wide_csv
3,baseline,50,0.782859,0.133687,wide_csv
4,lora_r4_stage2,50,0.780073,0.142953,wide_csv
5,flan_t5_small_prompt_0shot,50,0.427297,0.135950,per_model_csvs
6,flan_t5_small_prompt_3shot,50,0.426677,0.145595,per_model_csvs
7,flan_t5_small_prompt_5shot,50,0.402114,0.137224,per_model_csvs


,model,group,n,comet_mean,comet_std,input_sources
0,idiom_only_v1,idioms_test,25,0.762717,0.133694,wide_csv
1,lora_r16_stage2,idioms_test,25,0.732651,0.143304,wide_csv
2,baseline,idioms_test,25,0.732551,0.135191,wide_csv
3,lora_r8_stage2,idioms_test,25,0.732034,0.147128,wide_csv
4,lora_r4_stage2,idioms_test,25,0.727875,0.151279,wide_csv
5,flan_t5_small_prompt_0shot,idioms_test,25,0.381018,0.118532,per_model_csvs
6,flan_t5_small_prompt_5shot,idioms_test,25,0.371568,0.127203,per_model_csvs
7,flan_t5_small_prompt_3shot,idioms_test,25,0.363405,0.106758,per_model_csvs
8,idiom_only_v1,wmt_test,25,0.851892,0.089399,wide_csv
9,lora_r16_stage2,wmt_test,25,0.846875,0.094321,wide_csv


group,model,idioms_test,wmt_test
0,baseline,0.732551,0.833167
1,flan_t5_small_prompt_0shot,0.381018,0.473576
2,flan_t5_small_prompt_3shot,0.363405,0.489949
3,flan_t5_small_prompt_5shot,0.371568,0.432660
4,idiom_only_v1,0.762717,0.851892
5,lora_r16_stage2,0.732651,0.846875
6,lora_r4_stage2,0.727875,0.832272
7,lora_r8_stage2,0.732034,0.842295


input_group,model,per_model_csvs__idioms_test,per_model_csvs__wmt_test,wide_csv__idioms_test,wide_csv__wmt_test
0,baseline,NaN,NaN,0.732551,0.833167
1,flan_t5_small_prompt_0shot,0.381018,0.473576,NaN,NaN
2,flan_t5_small_prompt_3shot,0.363405,0.489949,NaN,NaN
3,flan_t5_small_prompt_5shot,0.371568,0.432660,NaN,NaN
4,idiom_only_v1,NaN,NaN,0.762717,0.851892
5,lora_r16_stage2,NaN,NaN,0.732651,0.846875
6,lora_r4_stage2,NaN,NaN,0.727875,0.832272
7,lora_r8_stage2,NaN,NaN,0.732034,0.842295


## 7. Sentence spread (useful for qualitative analysis)

In [23]:
# spread within each group across models
spread_df = (
    scored_df.pivot_table(
        index=["src", "ref", "group"],
        columns="model",
        values="comet",
        aggfunc="mean",
    )
    .reset_index()
)

model_cols = [c for c in spread_df.columns if c not in ["src", "ref", "group"]]
spread_df["comet_max"] = spread_df[model_cols].max(axis=1)
spread_df["comet_min"] = spread_df[model_cols].min(axis=1)
spread_df["comet_spread"] = spread_df["comet_max"] - spread_df["comet_min"]
spread_df = spread_df.sort_values(["group", "comet_spread"], ascending=[True, False]).reset_index(drop=True)

display(spread_df.head(10))

model,src,ref,group,baseline,flan_t5_small_prompt_0shot,flan_t5_small_prompt_3shot,flan_t5_small_prompt_5shot,idiom_only_v1,lora_r16_stage2,lora_r4_stage2,lora_r8_stage2,comet_max,comet_min,comet_spread
0,But before she could launch her charm offensiv...,Doch bevor sie ihre Charmeoffensive starten ko...,idioms_test,0.932659,0.252559,0.169645,0.169645,0.863792,0.932659,0.945956,0.932659,0.945956,0.169645,0.776311
1,"Bend the knee, peasant! Admit that I am your r...","Beuge das Knie, Bauer! Gib zu, dass ich dein r...",idioms_test,0.799448,0.347663,0.319701,0.282064,0.917172,0.799448,0.799448,0.799448,0.917172,0.282064,0.635108
2,"For the young Canadian servicemen, the order t...",Die jungen kanadischen Soldaten erwarteten täg...,idioms_test,0.812785,0.180694,0.199865,0.181853,0.681498,0.812785,0.812785,0.812785,0.812785,0.180694,0.632091
3,"""Regardless of their cause, inequalities are a...","""Unabhängig von ihrer Ursache sind Ungleichhei...",idioms_test,0.842750,0.290774,0.307880,0.317754,0.876986,0.827501,0.860689,0.827501,0.876986,0.290774,0.586211
4,We're cut from the same cloth as any American ...,Wir sind aus dem gleichen Holz geschnitzt wie ...,idioms_test,0.728234,0.336061,0.336061,0.336061,0.907209,0.848501,0.848501,0.848501,0.907209,0.336061,0.571148
5,"It would certainly be fun in an active family,...",Es wäre sicher lustig in einer aktiven Familie...,idioms_test,0.901928,0.376942,0.416186,0.367450,0.909396,0.898602,0.900598,0.898602,0.909396,0.367450,0.541946
6,"I wouldn't say they're like chalk and cheese, ...","Ich würde nicht sagen, dass sie wie Tag und Na...",idioms_test,0.775548,0.469741,0.241811,0.379842,0.775548,0.775548,0.775548,0.775548,0.775548,0.241811,0.533737
7,"Dad says I'm not cut out for the military, but...","Dad sagt, ich bin nicht fürs Militär gemacht, ...",idioms_test,0.924299,0.423057,0.404513,0.404513,0.928583,0.924299,0.924299,0.924299,0.928583,0.404513,0.524071
8,Despite several bilateral and multilateral mee...,Trotz mehrerer bilateraler und multilateraler ...,idioms_test,0.866841,0.404440,0.415179,0.409231,0.895262,0.867457,0.866841,0.867457,0.895262,0.404440,0.490822
9,I don't want to be the blind leading the blind.,"Ich möchte nicht der Blinde sein, der die Blin...",idioms_test,0.789665,0.329930,0.329930,0.307702,0.693753,0.789665,0.789665,0.789665,0.789665,0.307702,0.481963


## 8. Optional merge with BLEU / chrF

In [24]:
merged_df = try_merge_metrics(overall_df)
if merged_df is not None:
    display(merged_df.head())

,model,n,comet_mean,comet_std,input_sources,timestamp,run_name,split,bleu,chrf,...,max_new_tokens,freeze_encoder,train_size,dev_size,seed,n_eval,notes,metrics_source_file,n_shots,gen_kwargs
0,idiom_only_v1,50,0.807305,0.121235,wide_csv,2026-02-19T19:12:29-08:00,idiom_only_v1,idioms_test,44.363297,64.640647,...,128.0,False,800.0,100.0,42.0,100.0,stage1 idiom-only fine-tune,metrics.csv,NaN,NaN
1,idiom_only_v1,50,0.807305,0.121235,wide_csv,2026-02-19T19:14:16-08:00,idiom_only_v1,wmt_test,26.589856,58.217984,...,128.0,False,20000.0,2000.0,42.0,3003.0,stage1 idiom-only fine-tune (eval on WMT),metrics.csv,NaN,NaN
2,idiom_only_v1,50,0.807305,0.121235,wide_csv,2026-03-07T20:04:44-08:00,idiom_only_v1,idioms_test,44.363297,64.640647,...,128.0,False,800.0,100.0,42.0,100.0,stage1 idiom-only fine-tune,metrics.csv,NaN,NaN
3,idiom_only_v1,50,0.807305,0.121235,wide_csv,2026-03-07T20:06:28-08:00,idiom_only_v1,wmt_test,26.589856,58.217984,...,128.0,False,20000.0,2000.0,42.0,3003.0,stage1 idiom-only fine-tune (eval on WMT),metrics.csv,NaN,NaN
4,idiom_only_v1,50,0.807305,0.121235,wide_csv,2026-03-22T16:14:52-07:00,idiom_only_v1,idioms_test,44.363297,64.640647,...,128.0,False,800.0,100.0,42.0,100.0,stage1 idiom-only fine-tune,metrics.csv,NaN,NaN


## 9. Save outputs

In [25]:
combined_df.groupby(["group", "model"]).size()

group        model                     
idioms_test  baseline                      25
             flan_t5_small_prompt_0shot    25
             flan_t5_small_prompt_3shot    25
             flan_t5_small_prompt_5shot    25
             idiom_only_v1                 25
             lora_r16_stage2               25
             lora_r4_stage2                25
             lora_r8_stage2                25
wmt_test     baseline                      25
             flan_t5_small_prompt_0shot    25
             flan_t5_small_prompt_3shot    25
             flan_t5_small_prompt_5shot    25
             idiom_only_v1                 25
             lora_r16_stage2               25
             lora_r4_stage2                25
             lora_r8_stage2                25
dtype: int64

In [26]:
sentence_scores_path = os.path.join(OUT_DIR, "comet_sentence_scores.csv")
sentence_spread_path = os.path.join(OUT_DIR, "comet_sentence_spread.csv")
summary_overall_path = os.path.join(OUT_DIR, "comet_summary_overall.csv")
summary_by_group_path = os.path.join(OUT_DIR, "comet_summary_by_group.csv")
summary_by_input_source_path = os.path.join(OUT_DIR, "comet_summary_by_input_source.csv")
pivot_by_group_path = os.path.join(OUT_DIR, "comet_pivot_by_group.csv")
pivot_by_input_source_path = os.path.join(OUT_DIR, "comet_pivot_by_input_source.csv")
merged_path = os.path.join(OUT_DIR, "comet_merged_with_bleu_chrf.csv")

scored_df.to_csv(sentence_scores_path, index=False)
spread_df.to_csv(sentence_spread_path, index=False)
overall_df.to_csv(summary_overall_path, index=False)
by_group_df.to_csv(summary_by_group_path, index=False)
by_input_source_df.to_csv(summary_by_input_source_path, index=False)
pivot_by_group_df.to_csv(pivot_by_group_path, index=False)
pivot_by_input_source_df.to_csv(pivot_by_input_source_path, index=False)

if merged_df is not None:
    merged_df.to_csv(merged_path, index=False)

print("Saved:")
print(" -", sentence_scores_path)
print(" -", sentence_spread_path)
print(" -", summary_overall_path)
print(" -", summary_by_group_path)
print(" -", summary_by_input_source_path)
print(" -", pivot_by_group_path)
print(" -", pivot_by_input_source_path)
if merged_df is not None:
    print(" -", merged_path)

Saved:
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_sentence_scores.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_sentence_spread.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_summary_overall.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_summary_by_group.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_summary_by_input_source.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_pivot_by_group.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_pivot_by_input_source.csv
 - /content/drive/MyDrive/ds266_idiom_mt/qual_preds/comet_eval/comet_merged_with_bleu_chrf.csv
